# 코스피/코스닥 종목 분석 시스템

이 노트북은 pyKRX를 기반으로 코스피/코스닥 종목을 분석하는 시스템입니다.

## 분석 목표
- 코스피/코스닥 종목 필터링 (거래량 10만주 이상, 상장 3년 이상)
- 각 종목별 내부지표 및 외부지표 계산
- 데이터 저장 및 통합 분석

## 지표 구성
### 내부지표
- RSI, 52주 고가/저가, MA 비율, Golden Cross
- 볼린저밴드, ATR, ADX, 변동성
- 수익률, 상승/하락 여부

### 외부지표  
- 코스피 수익률, 주간수익률 순위
- Hurst 지수


## 1. 환경설정 및 라이브러리

In [ ]:
# 환경 설정 및 라이브러리 import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import os
import warnings
warnings.filterwarnings('ignore')

# pyKRX 관련
from pykrx import stock
from pykrx import bond

# 기술적 지표 계산
import talib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
import os

# Find the path to the font file
font_path = '~/Library/Fonts/NanumGothic.otf'
fm.fontManager.addfont(font_path)

# 2. Set the font family for Matplotlib
plt.rc('font', family='NanumGothic')

# 추가 한글 지원 설정
plt.rcParams['font.sans-serif'] = plt.rcParams['font.family']
plt.rcParams['axes.unicode_minus'] = False

# 디렉토리 생성
os.makedirs('./data', exist_ok=True)
os.makedirs('./analysis', exist_ok=True)

print("✅ 환경 설정 완료")
print(f"📁 데이터 저장 경로: {os.path.abspath('./data')}")
print(f"📁 분석 결과 저장 경로: {os.path.abspath('./analysis')}")


## 2. 데이터 로딩 함수

In [ ]:
def get_date_from_kospi():
    start_date = "19000101"
    today = datetime.now().strftime('%Y%m%d')
    kospi_df = get_index_data(start_date, today, "KOSPI")
    date_col = pd.to_datetime(kospi_df.index)
    date_min = date_col.min()
    date_max = date_col.max()
    print(f'Date Range(KOSPI): {date_min.strftime("%Y%m%d")} ~ {date_max.strftime("%Y%m%d")}')
    
    result = pd.DataFrame(date_col)
    result.to_csv('./data/features/kospi_date.csv', encoding='utf-8-sig')
    return result

def get_date_list(start_date="20000101", end_date=None):
    if end_date is None:
        end_date = datetime.now().strftime('%Y%m%d')
    dates = pd.date_range(start=start_date, end=end_date).strftime('%Y%m%d')
    date_list = [stock.get_nearest_business_day_in_a_week(date) for date in dates]
    date_list = list(dict.fromkeys(date_list))
    return date_list

def init_date_list():
    date_list = get_date_list()
    if not date_list:
        print("❌ 날짜 목록을 생성하지 못했습니다. (목록이 비어있음)")
        return []
        
    date_list_df = pd.DataFrame(date_list, columns=['date'])
    date_list_df.to_csv('./data/features/date_list.csv', encoding='utf-8-sig')
    print(f"📈 날짜 목록 저장 완료 ({date_list[0]} ~ {date_list[-1]})")
    return date_list

def read_date_list():
    date_list_df = pd.read_csv('./data/features/date_list.csv', index_col=0, parse_dates=True)
    date_list = date_list_df['date'].tolist()
    
    # 🌟 ERROR PREVENTION: Check if the list is empty
    if not date_list:
        print("❌ 날짜 목록 로드 실패. (목록이 비어있음)")
        return []

    print(f"📈 날짜 목록 로드 완료 ({date_list[0]} ~ {date_list[-1]})")
    return date_list

# init_date_list()
get_date_from_kospi()

In [ ]:
def load_data(start_date, end_date, ticker, dtype):
    print(f"📊 {ticker}: 데이터 로딩 중 ({start_date} ~ {end_date}), dtype: {dtype}")
    if dtype == "stock":
        return stock.get_market_ohlcv_by_date(start_date, end_date, ticker)
    elif dtype == "index":
        return stock.get_index_ohlcv(start_date, end_date, ticker)

def get_new_data(start_date, end_date, ticker, file_path, dtype):
    # 파일이 없거나 손상된 경우 새로 생성
    try:
        print(f"📊 {ticker}: 새 데이터 파일 생성 중 ({start_date} ~ {end_date})")
        df = load_data(start_date, end_date, ticker, dtype) 
        df.to_csv(file_path, encoding='utf-8-sig')
        print(f"📈 {ticker}: 데이터 저장 완료 ({start_date} ~ {end_date})")
        # 요청된 범위 반환
        print('NEW INDEX:', df.index)
        start_dt = pd.to_datetime(start_date, format='%Y%m%d')
        end_dt = pd.to_datetime(end_date, format='%Y%m%d')
        filtered_df = df[(start_dt <= df.index) & (df.index <= end_dt)]
        return filtered_df
            
    except Exception as e:
        print(f"❌ [{ticker}] ({start_date} ~ {end_date}): 데이터 가져오기 실패 - {str(e)}")
        return pd.DataFrame()

# 로컬 데이터 관리 함수들
def get_local_data(start_date, end_date, file_path, ticker, dtype="stock"):
    """
    로컬 파일에서 주식 데이터를 가져오는 함수
    - 파일이 없으면 새로 생성
    - 파일이 있으면 start_date ~ end_date 범위의 데이터 반환
    - end_date까지 데이터가 부족하면 자동으로 업데이트
    """
    
    if os.path.exists(file_path):
        try:
            # 기존 파일 읽기
            print(f"📈 [{ticker}]: 데이터 로딩 중 - {file_path}")
            df = pd.read_csv(file_path, index_col=0, parse_dates=True)
            start_dt = getNextDay(datetime.strptime(start_date, '%Y%m%d'))
            end_dt = getPreviousDay(datetime.strptime(end_date, '%Y%m%d'))

            if df.empty:
                print(f"{ticker} 데이터가 없습니다. 새로 가져옵니다.")
                return get_new_data(start_date, end_date, ticker, file_path, dtype)
            
            df.index = pd.to_datetime(df.index)
            first_dt = df.index[0]
            last_dt = df.index[-1]
            

            # end_date까지 데이터가 충분한지 확인
            if first_dt <= start_dt and end_dt <= last_dt:
                # 충분한 경우, 요청된 범위만 반환
                filtered_df = df[(df.index >= start_dt) & (df.index <= end_dt)]
                return filtered_df

            else:
                # 부족한 경우, 부족한 부분을 새로 가져와서 추가
                print(f"📈 {ticker}: 데이터 업데이트 중 ({first_dt} ~ {last_dt}) -> {start_dt} ~ {end_dt}")
                combined_df = [df]
                # 부족한 기간의 데이터 가져오기
                if end_dt > last_dt:
                    update_start = (last_dt + timedelta(days=1)).strftime('%Y%m%d')
                    update_end = end_dt.strftime('%Y%m%d')
                    combined_df.append(load_data(update_start, update_end, ticker, dtype))
                    
                if start_dt < first_dt:
                    update_start = start_dt.strftime('%Y%m%d')
                    update_end = (first_dt - timedelta(days=1)).strftime('%Y%m%d')
                    combined_df.insert(0, load_data(update_start, update_end, ticker, dtype))
                    
                if len(combined_df) > 1:
                    combined_df = pd.concat(combined_df)
                    combined_df = combined_df[~combined_df.index.duplicated(keep='last')]  # 중복 제거
                    # combined_df.to_csv(file_path, encoding='utf-8-sig')
                    combined_df.to_csv(file_path)
                else:
                    combined_df = combined_df[0]
                
                filtered_df = combined_df[(combined_df.index >= start_dt) & (combined_df.index <= end_dt)]
                return filtered_df
                    
        except Exception as e:
            print(f"❌ {ticker}: 파일 읽기 오류 - {str(e)}")
            # 파일이 손상된 경우 새로 생성
            pass
    else:
        return get_new_data(start_date, end_date, ticker, file_path, dtype)
    

def get_stock_data(start_date, end_date, ticker):
    file_path = f"./data/{ticker}.csv"
    result = get_local_data(start_date, end_date, file_path, ticker, dtype="stock")
    return result
    

def get_index_data(start_date, end_date, index_code):
    index_code_map = {
        'KOSPI': "1001",
        'KOSDAQ': "2001",
        'KOSPI200': "1002",
        'KOSPI100': "1003",
        'KOSPI50': "1004",
        'KOSPI10': "1005",
        'KOSPI5': "1006",
    }
    ticker = index_code_map[index_code]
    file_path = f"./data/index/{ticker}_{index_code}.csv"
    return get_local_data(start_date, end_date, file_path, ticker, dtype="index")

def init_date_list():
    global date_list
    global init_completed
    start_date = "19750104"
    today = datetime.now().strftime('%Y%m%d')
    kospi_df = get_index_data(start_date, today, "KOSPI")
    date_col = pd.to_datetime(kospi_df.index)
    date_min = date_col.min()
    date_max = date_col.max()
    print('initializing date list...')
    print(f'Date Range(KOSPI): {date_min.strftime("%Y%m%d")} ~ {date_max.strftime("%Y%m%d")}')
    date_list = date_col
    init_completed = True

def getNextDay(date):
    while not isDateInList(date):
        date = (date + timedelta(days=1))
    return date

def getPreviousDay(date):
    while not isDateInList(date):
        date = (date - timedelta(days=1))
    return date

def isDateInList(date):
    if init_completed:
        return date in date_list
    else:
        return True
    
init_completed = False
init_date_list()
print("✅ 로컬 데이터 관리 함수 정의 완료")


#### 2-a. 데이터 로딩 테스트

In [ ]:
codes = stock.get_market_ticker_list(market="KOSPI")[:3]
names = [stock.get_market_ticker_name(code) for code in codes ]
print(codes, names)

def test():
    ticker = '095570'
    start_date = "20200101"
    end_date = "20201231"
    market = "KOSPI"
    file_path = f"./data/{ticker}.csv"
    
    df = get_stock_data(start_date, end_date, ticker)
    # df2 = get_index_data(start_date, end_date, market)
    # merged_df = pd.concat([df, df2], axis=1)
    display(df)
    # display(df2)
    # display(merged_df)

test()
    

In [ ]:
def test():
    ticker = '095570'
    today = datetime.now()
    recent_start = (today - timedelta(days=365)).strftime('%Y%m%d')
    end_date = today.strftime('%Y%m%d')
    df = stock.get_market_ohlcv_by_date(recent_start, end_date, ticker)
    avg_volume = df['거래량'].mean()
    recent_volume = df['거래량'].tail(20).mean()  # 최근 20일 평균
    print(f'Ticker: {ticker}')
    print(f'Average Volume: {avg_volume}')
    print(f'Recent Volume: {recent_volume}')
    print(f'Data Length: {len(df)}')

test()

## 3. 코스피/코스닥 종목 로딩

In [ ]:
# 코스피/코스닥 종목 불러오기 및 필터링
def get_filtered_stocks():
    """
    코스피/코스닥 종목을 불러와서 필터링
    - 거래량 10만주 이상 (최근 또는 평균)
    - 상장 후 3년 이상
    """
    print("📊 코스피/코스닥 종목 불러오는 중...")
        
    # 오늘 날짜
    today = datetime.now()
    end_date = today.strftime('%Y%m%d')
    start_date = "19900101" #19750104 이후의 날짜여야함.
    # recent_start = (today - timedelta(days=730)).strftime('%Y%m%d')

    # 코스피 종목 리스트
    kospi_tickers = stock.get_market_ticker_list(market="KOSPI")
    kosdaq_tickers = stock.get_market_ticker_list(market="KOSDAQ")

    print(f"📈 코스피 종목 수: {len(kospi_tickers)}")
    print(f"📈 코스닥 종목 수: {len(kosdaq_tickers)}")

    all_tickers = []
    filtered_stocks = []

    # 모든 종목 정보 수집
    for market, tickers in [("KOSPI", kospi_tickers), ("KOSDAQ", kosdaq_tickers)]:
        for ticker in tickers:
            try:
                # 종목명 가져오기
                name = stock.get_market_ticker_name(ticker)
                all_tickers.append({
                    'ticker': ticker,
                    'name': name,
                    'market': market
                })
            except:
                continue

    print(f"📋 전체 종목 수: {len(all_tickers)}")

    def check_stock_info(ticker):
        df = get_stock_data(start_date, end_date, stock_info['ticker'])
        if df.empty or len(df) < 500:  # 최소 500일 데이터 필요
            return False
            
        # 거래량 필터링 (10만주 이상)
        avg_volume = df['거래량'].mean()
        recent_volume = df['거래량'].tail(20).mean()  # 최근 20일 평균
        
        if avg_volume < 100000 and recent_volume < 100000:
            return False

        result = {
            'ticker': stock_info['ticker'],
            'name': stock_info['name'],
            'market': stock_info['market'],
            'avg_volume': avg_volume,
            'recent_volume': recent_volume,
            'data_length': len(df)
        }
        
        print(f'{stock_info["ticker"]} {stock_info["name"]} {stock_info["market"]} {avg_volume} {recent_volume} {len(df)}')
        return result
        

    # 필터링 적용
    print("🔍 필터링 적용 중...")
    num = 0
    for stock_info in all_tickers:
        try:
            # 최근 1년 데이터로 거래량 확인
            num += 1
            print(f'{num} / {len(all_tickers)}, {len(filtered_stocks)}개 완료')
            result = check_stock_info(stock_info)
            if result:
                filtered_stocks.append(result)
            
        except Exception as e:
            continue

    print(f"✅ 필터링 완료: {len(filtered_stocks)}개 종목")

    # 결과를 DataFrame으로 변환
    filtered_df = pd.DataFrame(filtered_stocks)

    if not filtered_df.empty:
        print(f"\n📊 필터링 결과 요약:")
        print(f"- 평균 거래량: {filtered_df['avg_volume'].mean():,.0f}주")
        print(f"- 최근 거래량: {filtered_df['recent_volume'].mean():,.0f}주")
        print(f"- 평균 데이터 길이: {filtered_df['data_length'].mean():.0f}일")
        
        print(f"\n🏆 상위 10개 종목 (거래량 기준):")
        top_stocks = filtered_df.nlargest(10, 'avg_volume')[['ticker', 'name', 'market', 'avg_volume']]
        print(top_stocks.to_string(index=False))

    return filtered_df

# 종목 필터링 실행
filtered_stocks_df = get_filtered_stocks()


## 4. 데이터 지표 계산

In [ ]:
def calculate_hurst_exponent(returns): # Hurst 지수 계산 (R/S 분석)
    n = len(returns)
    if n < 10:
        return np.nan
    
    # 다양한 구간 크기
    sizes = np.logspace(0.5, np.log10(n//4), 10, dtype=int)
    sizes = np.unique(sizes)
    rs_values = []
    
    for size in sizes:
        if size < 2:
            continue
            
        # 구간별 R/S 계산
        rs_list = []
        for i in range(0, n - size + 1, size):
            segment = returns[i:i+size]
            if len(segment) < size:
                continue
                
            mean_segment = np.mean(segment)
            adjusted_segment = segment - mean_segment
            cumulative_deviation = np.cumsum(adjusted_segment)
            R = np.max(cumulative_deviation) - np.min(cumulative_deviation)
            S = np.std(segment, ddof=0)
            if S > 0:
                rs_list.append(R / S)
        
        if rs_list:
            rs_values.append(np.mean(rs_list))
    
    if len(rs_values) < 2:
        return np.nan

    # 로그 회귀로 Hurst 지수 계산
    log_sizes = np.log(sizes[:len(rs_values)])
    log_rs = np.log(rs_values)
    # 선형 회귀
    coeffs = np.polyfit(log_sizes, log_rs, 1)
    hurst = coeffs[0]
    return hurst

print("✅ 외부지표 계산 함수 정의 완료")

def classify_value(x, threshold=0.1):
    return np.where(x >= threshold, 1, np.where(x <= -threshold, -1, 0 ))

# 내부지표 계산 함수들
def calculate_stock_indicators(df):
    """
    내부지표 계산
    - RSI, 52주 고가/저가, MA 비율, Golden Cross
    - 볼린저밴드, ATR, ADX, 변동성
    - 수익률, 상승/하락 여부
    """
    
    for col in ['시가', '저가', '고가']:
        if col not in df.columns:
            df[col] = df['종가']
        df[col] = np.where(df[col]==0,df['종가'],df[col])
    
    result = pd.DataFrame()
    # # 1. RSI (14일)
    result['RSI'] = talib.RSI(df['종가'], timeperiod=14)
    high_52w = df['고가'].rolling(window=252).max()
    low_52w = df['저가'].rolling(window=252).min()
    # 고가, 저가가 아닌 종가를 기준으로 max, min을 계산하는 것도 고려해야 함.
    # 252-1 개의 데이터가 소거됨.
    result['52주_고가_대비_주가'] = (df['종가'] / high_52w)
    result['52주_저가_대비_주가'] = (df['종가'] / low_52w)
    
    # 3. 이동평균 비율
    result['close'] = df['종가']
    result['ma5'] = talib.SMA(df['종가'], timeperiod=5)
    result['ma20'] = talib.SMA(df['종가'], timeperiod=20)
    result['ma60'] = talib.SMA(df['종가'], timeperiod=60)
    
    result['MA60_MA20'] = result['ma60'] / result['ma20']
    result['MA20_MA5'] = result['ma20'] / result['ma5']
    result['MA5_MA1'] = result['ma5'] / result['close']
    
    # Golden Cross (MA20 > MA60)
    result['ma20_ma60'] = result['ma20'] > result['ma60']

    current_cross_up = (result['ma20'] > result['ma60'])
    previous_no_cross = (result['ma20'].shift(1) <= result['ma60'].shift(1))
    result['Golden_Cross'] = np.where(current_cross_up & previous_no_cross, 1, 0)
    
    # 4. 볼린저밴드 (20일, 표준편차 2, ddof=0)
    result['bb_upper'], result['bb_middle'], result['bb_lower'] = talib.BBANDS(df['종가'], timeperiod=20, nbdevup=2, nbdevdn=2, matype=0)
    result['bb_width'] = (result['bb_upper'] - result['bb_lower']) / result['bb_middle']
    
    # 5. ATR (Average True Range)
    result['atr'] = talib.ATR(df['고가'], df['저가'], df['종가'], timeperiod=14)
    
    # 6. ADX (Average Directional Index)
    result['adx'] = talib.ADX(df['고가'], df['저가'], df['종가'], timeperiod=14)
    
    # 7. 수익률 계산 (로그 수익률)
    result['daily_return'] = np.log(df['종가'] / df['종가'].shift(1))
    result['weekly_return'] = np.log(df['종가'] / df['종가'].shift(5))
    result['20week_return'] = np.log(df['종가'] / df['종가'].shift(100)) # 20주 수익률 (100일)
    
    # 8. 상장 후 기간 (연도)
    result['years'] = len(df) / 252  # 거래일 기준
    volume_mean, volume_std = np.mean(df['거래량']), np.std(df['거래량'])
    result['volume(std)'] = (df['거래량'] - volume_mean) / volume_std
    result['volatility'] = result['daily_return'].rolling(window=20).std() * np.sqrt(20)
    
    result['weekly_up'] = classify_value(result['weekly_return'], 0.01)
    result['20week_up'] = classify_value(result['20week_return'], 0.05)
    result['weekly_up_count'] = result['weekly_up'].rolling(window=5).apply(lambda x: (x == 1).sum())
    result['20week_up_count'] = result['weekly_up'].rolling(window=20).apply(lambda x: (x == 1).sum())
    result['hurst_exponent'] = calculate_hurst_exponent(result['daily_return'])
    
    return result

def calculate_index_indicators(df):
    for col in ['시가', '저가', '고가']:
        if col not in df.columns:
            df[col] = df['종가']
        df[col] = np.where(df[col]==0,df['종가'],df[col])
    
    result = pd.DataFrame()
    high_52w = df['고가'].rolling(window=50).max()
    low_52w = df['저가'].rolling(window=50).min()
    result['close'] = df['종가']
    result['weekly_return'] = df['종가']/df['종가'].shift(5)
    result['20week_return'] = df['종가']/df['종가'].shift(20)
    result['10주_고가_대비_주가'] = df['종가'] / high_52w
    result['10주_저가_대비_주가'] = df['종가'] / low_52w
    result['RSI'] = talib.RSI(df['종가'], timeperiod=14)
    result['volatility'] = df['종가'].rolling(window=20).std() * np.sqrt(20)
    result.columns = [f'index_{col}' for col in result.columns]

    return result

print("✅ 내부지표 계산 함수 정의 완료")


In [ ]:
def analyze_index(start_date, end_date, market):    
    df = get_index_data(start_date, end_date, market)
    result = calculate_index_indicators(df)
    return result

def init_analysis_index():
    index_info = [
        {
        'market': "KOSPI",
        "ticker": "1001",
        'start_date': "19750104",
        }, {
            'market': "KOSDAQ",
            "ticker": "2001",
            'start_date': "19960701",
        }
    ]
    end_date = datetime.now().strftime('%Y%m%d')

    for info in index_info:
        df = analyze_index(info['start_date'], end_date, info['market'])
        df.to_csv(f"./analysis/index/{info['market']}.csv", encoding='utf-8-sig')

def get_analysis_index(start_date, end_date, market):
    try:
        # 기존 파일 읽기
        file_path = f"./analysis/index/{market}.csv"
        df = pd.read_csv(file_path, index_col=0, parse_dates=True)
        df.index = pd.to_datetime(df.index)
        
        if df.empty:
            print(f"{market} 데이터가 없습니다. 새로 가져옵니다.")
            return get_new_data(start_date, end_date, market, file_path, dtype)

        start_dt = getNextDay(pd.to_datetime(start_date, format='%Y%m%d'))
        end_dt = getPreviousDay(pd.to_datetime(end_date, format='%Y%m%d'))

        first_dt = df.index[0]
        last_dt = df.index[-1]
        
        # end_date까지 데이터가 충분한지 확인
        if last_dt >= end_dt and first_dt <= start_dt:
            # 충분한 경우, 요청된 범위만 반환
            filtered_df = df[(df.index >= start_dt) & (df.index <= end_dt)]
            return filtered_df
        else:
            # 부족한 경우, 부족한 부분을 새로 가져와서 추가
            print(f"📈 {market}: 데이터 업데이트 중 ({first_dt} ~ {last_dt}) -> ({start_dt} ~ {end_dt}")
            combined_df = [df]

            # 부족한 기간의 데이터 가져오기
            if last_dt < end_dt:
                update_start = (last_dt + timedelta(days=1)).strftime('%Y%m%d')
                update_end = end_dt.strftime('%Y%m%d')
                combined_df.append(analysis_index(), update_end, market, 'index')
            
            if start_dt < first_dt:
                update_start = start_dt.strftime('%Y%m%d')
                update_end = (first_dt - timedelta(days=1)).strftime('%Y%m%d')
                combined_df.insert(0, load_data(update_start, update_end, market, 'index'))
            
            if len(combined_df) > 1:
                combined_df = pd.concat(combined_df)
                combined_df = combined_df[~combined_df.index.duplicated(keep='last')]  # 중복 제거
                # combined_df = combined_df.sort_index()
                combined_df.to_csv(file_path, encoding='utf-8-sig')
            else:
                combined_df = combined_df[0]
            # 요청된 범위 반환
            filtered_df = combined_df[(combined_df.index >= start_dt) & (combined_df.index <= end_dt)]
            return filtered_df
                
    except Exception as e:
        print(f"❌ {market}: 파일 읽기 오류 - {str(e)}")
        # 파일이 손상된 경우 새로 생성
        pass
    
    # 파일 저장은 하지 않음. 필요할 때 만들어 쓰기.
    # 파일 만들거면 init_analysis_index 호출
    file_path = f"./analysis/index/{market}.csv"
    result = analyze_index(start_date, end_date, market)
    return result

# init_analysis_index()


### a. 데이터 테스트

In [ ]:
def get_test_df():
    ticker = '095570'
    today = datetime.now()
    recent_start = (today - timedelta(days=1080)).strftime('%Y%m%d')
    end_date = today.strftime('%Y%m%d')
    df = get_stock_data(recent_start, end_date, ticker)
    return df

def test():
    df = get_test_df()
    result_df = calculate_stock_indicators(df)
    display(result_df)
test()

## 5. 분석 파이프라인

#### 각 종목 별로 데이터를 불러와 분석합니다

In [ ]:
get_analysis_index("19900101", "20220101", "KOSDAQ")
# analyze_single_stock("060310", "3S", "KOSDAQ", "19900101", "20220101")

In [ ]:
# 종목별 데이터 로딩 및 분석 파이프라인
def analyze_single_stock(ticker, name, market, start_date, end_date):
    """
    단일 종목 분석
    """
    try:
        print(f"📊 {name}({ticker}) 분석 중...")
        
        # 1. 데이터 로딩
        savepath = f"./analysis/{ticker}_{name.replace('/', '_')}.csv"
        df = get_stock_data(start_date, end_date, ticker)
        index_df = get_analysis_index(start_date, end_date, market)
        df = df.join(index_df, how='inner')
        # df = pd.concat([df, index_df], axis=1)
        # 2. 내부지표 계산
        stock_indicators = calculate_stock_indicators(df)
        stock_indicators['diff_with_index'] = stock_indicators['weekly_return'] - index_df['index_weekly_return']
        stock_indicators.to_csv(savepath, encoding='utf-8-sig')
        
        # 4. 결과 통합
        result = {
            'ticker': ticker,
            'name': name,
            'market': market,
            'analysis_date': datetime.now().strftime('%Y-%m-%d'),
            'result_path': savepath
        }
        
        print(f"✅ {name}: 분석 완료")
        return result
        
    except Exception as e:
        print(f"❌ {name}: 오류 발생 - {str(e)}")
        return None

def run_analysis_pipeline(filtered_stocks_df, start_date, end_date):
    """
    전체 분석 파이프라인 실행
    """
    print("🚀 분석 파이프라인 시작")
    print(f"📅 분석 기간: {start_date} ~ {end_date}")
    print(f"📊 분석 대상: {len(filtered_stocks_df)}개 종목")
    
    # 코스피 지수 데이터 로딩
    print("📈 코스피 지수 데이터 로딩 중...")
    init_analysis_index()
    
    # 종목별 분석 실행
    results = []
    total_stocks = len(filtered_stocks_df)
    
    for idx, row in filtered_stocks_df.iterrows():
        ticker = row['ticker']
        name = row['name']
        market = row['market']
        
        print(f"\n[{idx+1}/{total_stocks}] {name}({ticker}) - {market}")
        result = analyze_single_stock(ticker, name, market, start_date, end_date)
        
        if result is not None:
            results.append(result)
        
        # 진행률 표시
        if (idx + 1) % 10 == 0:
            print(f"📊 진행률: {idx+1}/{total_stocks} ({(idx+1)/total_stocks*100:.1f}%)")
    
    print(f"\n🎉 분석 완료: {len(results)}개 종목 성공")
    return results


#### 분석 테스트

In [ ]:

# 분석 실행 (테스트용 - 상위 5개 종목만)
if not filtered_stocks_df.empty:
    # 분석 기간 설정 (최근 3년)
    today = datetime.now()
    end_date = today.strftime('%Y%m%d')
    start_date = (today - timedelta(days=365*3)).strftime('%Y%m%d')
    
    # 테스트용으로 상위 5개 종목만 분석
    test_stocks = filtered_stocks_df[filtered_stocks_df['market'] == 'KOSDAQ'].head(5)
    print(f"🧪 테스트 모드: 상위 {len(test_stocks)}개 종목 분석")
    
    analysis_results = run_analysis_pipeline(test_stocks, start_date, end_date)
else:
    print("❌ 필터링된 종목이 없습니다.")
    analysis_results = []

#### 전체종목 분석

- 전체 필터링된 종목에 대해 분석 실행
- 주의: 시간이 오래 걸릴 수 있습니다.

In [ ]:
# 전체 종목 분석 실행 
def run_full_analysis():
    """
    """
    if filtered_stocks_df.empty:
        print("❌ 필터링된 종목이 없습니다.")
        return
    
    print("⚠️ 전체 종목 분석을 시작합니다.")
    print(f"📊 분석 대상: {len(filtered_stocks_df)}개 종목")
    print("⏰ 예상 소요 시간: 10-30분 (종목 수에 따라)")
    
    # 분석 기간 설정
    today = datetime.now()
    end_date = today.strftime('%Y%m%d')
    # start_date = (today - timedelta(days=365*3)).strftime('%Y%m%d')
    start_date = "19800101"
    
    # 전체 분석 실행
    full_analysis_results = run_analysis_pipeline(filtered_stocks_df, start_date, end_date)
    
    if full_analysis_results:
        # 통합 분석 결과 생성
        print(f"\n🎉 전체 분석 완료!")
        print(f"📊 성공한 종목: {len(full_analysis_results)}개")
        print(f"💾 결과 저장 위치:")
    else:
        print("❌ 분석 결과가 없습니다.")

# 전체 분석 실행 함수 (필요시 주석 해제)
full_results = run_full_analysis()
